In [1]:
import cdsapi
import os
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import as_completed

In [2]:
load_dotenv()

URL="https://cds.climate.copernicus.eu/api"
KEY=os.getenv("CDS_API_KEY_MAI")

In [3]:
timetable = ["00:00", "03:00", "06:00", "09:00", "12:00", "15:00","18:00", "21:00"]
year_array = [str(i) for i in range(1980, 2026)]
month_array = [str(i).rjust(2, '0') for i in range(1, 13)]
day_array = [str(i).rjust(2, '0') for i in range(1, 32)]

vars = "850hPa_vertical_velocity"
dataset = "reanalysis-era5-single-levels"

In [4]:
def get_climate_data_time_based(vars, year_array, month_array, day_array, time):
    request = {
        "product_type": ["reanalysis"],
        "variable": [vars],
        "year": year_array,
        "month": month_array,
        "day": day_array,
        "time": [time],
        "data_format": "netcdf",
        "download_format": "zip",
        "area": [21, 103, 10, 110]
    }

    client = cdsapi.Client(url=URL, key=KEY)
    client.retrieve(dataset, request, target=f"./era5_{vars}_vn_{time[:2]}.nc")

In [ ]:
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(get_climate_data_time_based, vars, year_array, month_array, day_array, time): time for time in timetable}

for future in as_completed(futures, timeout=20):
    time = futures[future]
    try:
        future.result()
        print(f"{time} - done")
    except Exception as e:
        print(f"{time} - failed: {e}")

2026-03-26 23:49:39,244 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-26 23:49:39,245 INFO Request ID is 7ba53a52-95b7-491c-b5fa-a1f552a486f3
2026-03-26 23:49:39,426 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus